In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
import pandas as pd
import rpy2.rinterface_lib.sexp as SEXP
from rpy2.robjects.vectors import ListVector
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations
import glob

In [ ]:
parameter_translation_map = {
    "A_viewerServerSelectionDistribution": "SP",
    "B_maxReservation": "MR [B]",
    "B_existingViewerServerSelectionDistribution": "ESP",
    "B_newViewerServerParentSelectionDistribution": "OSP",
    "B_existingViewerServerSelectionTreeTraversal": "EST",
    "B_newViewerServerParentSelectionTreeTraversal": "OST",
    "C_maxReservation": "MR [C]",
    "C_reservationType": "RT",
    "C_alreadyLinkedTreeTraversal": "EVST",
    "C_alreadyLinkedDistribution": "EVSP",
    "C_linkedSpaceTreeTraversal": "OVST",
    "C_linkedSpaceDistribution": "OVSP",
    "C_viewerAndLinkFromDistribution": "NVSP",
    "C_publisherServerSelectionDistribution": "PSP",
    "maxReservation": "MR",
}
distribution_cols = ["SP", "ESP", "OSP", "EVSP", "OVSP", "NVSP", "PSP"]
tree_traversal_cols = ["EST", "OST", "EVST", "OVST"]
distribution_translation_map = {
    "LOW_LOAD": "LL",
    "HIGH_LOAD": "HL",
    "ROUND_ROBIN": "RR",
    "RANDOM": "RND",
}
tree_traversal_translation_map = {
    "TOP_TO_BOTTOM": "TTB",
    "BOTTOM_TO_TOP": "BTT",
    "IGNORE_TREE_LEVEL": "IT",
}
reservation_type_translation_map = {
    "RESERVATION_PER_SERVER": "SER",
    "RESERVATION_PER_SESSION": "SES",
}
alg_a_parameters = ["SP"]
alg_b_parameters = ["MR [B]", "ESP", "OSP", "EST", "OST", "MR"]
alg_c_parameters = ["MR [C]", "RT", "EVST", "EVSP", "OVST", "OVSP", "NVSP", "PSP", "MR"]

In [ ]:
def remove_duplicates(configs, training_exps, testing_exps):
    # Some configs are duplicate of others with more experiments in different instances, we want to keep the most informative ones
    duplicate_configs = configs[configs.duplicated(keep=False)]
    # if a row contains a RND parameter, we want to remove those "duplicate" configs from our true duplicates
    duplicate_configs = duplicate_configs[
        ~duplicate_configs.apply(
            lambda row: any(row[col] == "RND" for col in duplicate_configs.columns),
            axis=1,
        )
    ]
    # get the indices of the rows that are duplicates in tuples, a tuple for each duplicate row of the same config, to be able to keep the one with the most experiments
    # Group duplicate rows by their values (excluding the index)
    duplicate_groups = {}
    for idx, row in duplicate_configs.iterrows():
        # Convert all values to string for consistent comparison (including NaN/None)
        row_key = frozenset(
            (col, str(row[col]) if pd.notna(row[col]) else "NaN")
            for col in duplicate_configs.columns
        )
        if row_key not in duplicate_groups:
            duplicate_groups[row_key] = []
        duplicate_groups[row_key].append(idx)

    # Only keep groups with more than one duplicate
    duplicate_indices_tuples = [
        tuple(indices) for indices in duplicate_groups.values() if len(indices) > 1
    ]

    best_cols = []
    # In training_exps, for each tuple, find the row with the most experiments
    for indices in duplicate_indices_tuples:
        # Get the rows corresponding to the indices
        cols = training_exps[list(indices)]
        # Find the column with the lower number of Nones, unless it is in testing_exps columns in which case we keep the one in testing_exps
        if testing_exps.columns.isin(indices).any():
            # If one of the indices is in testing_exps, we want to keep that one
            best_col = next(
                (idx for idx in indices if idx in testing_exps.columns), None
            )
        else:
            non_na_counts = cols.notna().count()
            best_col = non_na_counts.idxmax()
        best_cols.append(best_col)
        # For that column, if another column has a value in a place where the best column has None, we want to keep that value
        for idx in indices:
            if idx != best_col:
                for row in training_exps.index:
                    if pd.isna(training_exps.at[row, best_col]) and not pd.isna(
                        training_exps.at[row, idx]
                    ):
                        training_exps.at[row, best_col] = training_exps.at[row, idx]
    # Now we can drop the duplicate rows from configs and training_exps that are not the best ones
    diff = duplicate_configs.index.difference(best_cols)
    new_configs = configs.drop(diff)
    new_training_exps = training_exps.drop(diff, axis=1)
    return new_configs, new_training_exps


def calc_dev(full_exps, dev):
    for idx, row in full_exps.iterrows():
        min_value = row.min()
        dev.loc[idx] = row.apply(
            lambda x, minimum_value=min_value: (
                100 * (x - minimum_value) / ((x + minimum_value) / 2)
                if pd.notna(x)
                else x
            )
        )


irace = importr("irace")


def load_irace_elite_configs(file_location):
    ro.r["load"](file_location)
    objects = ro.r["iraceResults"]
    testing = ListVector(objects.rx2("testing"))
    test_exp_matrix = testing.rx2("experiments")
    test_matrix_row_names = list(test_exp_matrix.rownames)
    test_matrix_col_names = list(test_exp_matrix.colnames)
    training_exps = objects.rx2("experiments")
    training_exps_col_names = list(training_exps.colnames)
    with (ro.default_converter + pandas2ri.converter).context():
        instances = irace.get_instanceID_seed_pairs(objects, instances=True)
        test_exp_matrix = pandas2ri.rpy2py(test_exp_matrix)
        test_exp_matrix = pd.DataFrame(
            test_exp_matrix, index=test_matrix_row_names, columns=test_matrix_col_names
        )
        training_exps = pandas2ri.rpy2py(training_exps)
        training_exps = pd.DataFrame(
            training_exps,
            index=instances["instanceID"],
            columns=training_exps_col_names,
        )
        training_exps = training_exps.sort_index()
        configs = objects.rx2("allConfigurations")
        configs = configs.replace(-2147483648, None)
        for col in configs.columns:
            configs[col] = configs[col].apply(
                lambda x: None if isinstance(x, SEXP.NACharacterType) else x
            )
        configs = configs.drop(columns=[".ID.", ".PARENT."])
        configs = configs.rename(columns=parameter_translation_map)
        for col in configs.select_dtypes(include=["float"]):
            if (configs[col] % 1 == 0).all():
                configs[col] = configs[col].astype("Int64")
        for col in configs.columns:
            if col in distribution_cols:
                configs[col] = configs[col].map(distribution_translation_map)
            if col in tree_traversal_cols:
                configs[col] = configs[col].map(tree_traversal_translation_map)
            if col == "RT":
                configs[col] = configs[col].map(reservation_type_translation_map)
        configs, training_exps = remove_duplicates(
            configs, training_exps, test_exp_matrix
        )
        elites = objects.rx2("allElites")[-1]
        elites = [str(e) for e in elites]
        elite_configs = configs.loc[elites]
        elite_configs = elite_configs.dropna(axis=1, how="all")
        for col in elite_configs.select_dtypes(include=["float"]):
            if (elite_configs[col] % 1 == 0).all():
                elite_configs[col] = elite_configs[col].astype("Int64")

        all_exps = pd.concat([training_exps, test_exp_matrix], axis=0)
        # coming from all_exps, for every row, in each cell change the value for the deviation from the best config
        dev = all_exps.copy()
        calc_dev(all_exps, dev)
        return configs, elite_configs, test_exp_matrix, training_exps, all_exps, dev


configs_map = {}
elite_configs_map = {}
testing_exps_map = {}
training_exps_map = {}
full_experiments_map = {}
deviation_map = {}
for root, _, files in os.walk("irace_results"):
    for file in files:
        if file.endswith("irace.Rdata"):
            file_path = os.path.join(root, file)
            dir_path = os.path.dirname(file_path)
            splitted = dir_path.split(os.sep)
            budget = splitted[1].split("_")[0]
            instance_type = splitted[2]
            server_capacity = splitted[3]
            try:
                configs, elite_configs, testing_exps, full_exps, all_exps, dev = (
                    load_irace_elite_configs(file_path)
                )
                # Ensure 'small' comes before 'medium' in the map
                if instance_type not in configs_map:
                    configs_map[instance_type] = {}
                configs_map[instance_type][server_capacity] = configs
                if instance_type not in elite_configs_map:
                    elite_configs_map[instance_type] = {}
                elite_configs_map[instance_type][server_capacity] = elite_configs
                if instance_type not in testing_exps_map:
                    testing_exps_map[instance_type] = {}
                testing_exps_map[instance_type][server_capacity] = testing_exps
                if instance_type not in training_exps_map:
                    training_exps_map[instance_type] = {}
                training_exps_map[instance_type][server_capacity] = full_exps
                if instance_type not in full_experiments_map:
                    full_experiments_map[instance_type] = {}
                full_experiments_map[instance_type][server_capacity] = all_exps
                if instance_type not in deviation_map:
                    deviation_map[instance_type] = {}
                deviation_map[instance_type][server_capacity] = dev
            except Exception as e:
                print(f"Failed to process {file_path}: {e}")


# Reorder the maps so 'small' comes first, then 'medium', and server capacities are sorted numerically
def reorder_instance_types_and_capacities(d):
    ordered = {}
    for instance in ["small", "medium"]:
        if instance in d:
            # Sort server capacities numerically
            capacities = d[instance]
            ordered_capacities = dict(
                sorted(capacities.items(), key=lambda x: int(x[0]))
            )
            ordered[instance] = ordered_capacities
    for instance in d:
        if instance not in ordered:
            capacities = d[instance]
            ordered_capacities = dict(
                sorted(capacities.items(), key=lambda x: int(x[0]))
            )
            ordered[instance] = ordered_capacities
    return ordered


configs_map = reorder_instance_types_and_capacities(configs_map)
elite_configs_map = reorder_instance_types_and_capacities(elite_configs_map)
testing_exps_map = reorder_instance_types_and_capacities(testing_exps_map)
training_exps_map = reorder_instance_types_and_capacities(training_exps_map)
full_experiments_map = reorder_instance_types_and_capacities(full_experiments_map)
deviation_map = reorder_instance_types_and_capacities(deviation_map)

In [ ]:
best_a_map = {}
best_b_map = {}
best_config_ids_map = {}


def cleanup_a_errors(full_exps, devs):
    full_exps.replace(1e15, np.nan, inplace=True)
    devs.where(full_exps.notna(), inplace=True)


display("Top A and B configurations per instance type and server capacity:")
for instance_type, server_capacities in full_experiments_map.items():

    for server_capacity, full_exps in server_capacities.items():
        configs = configs_map[instance_type][server_capacity]
        devs = deviation_map[instance_type][server_capacity]


        algo_ids = {
            algo: configs[configs["algorithm"] == algo].index.astype(str).tolist()
            for algo in configs["algorithm"].unique()
        }

        cleanup_a_errors(full_exps, devs)

        # Extract A and B experiment and deviation data
        a_devs = devs[algo_ids["A"]]
        b_devs = devs[algo_ids["B"]]

        # Compute mean deviations, skipping NaNs, and sort
        a_devs_mean = a_devs.mean(axis=0).sort_values()


        b_devs_mean = b_devs.mean(axis=0).sort_values()
        if instance_type not in best_a_map:
            best_a_map[instance_type] = {}
        if instance_type not in best_b_map:
            best_b_map[instance_type] = {}
        best_a_map[instance_type][server_capacity] = a_devs_mean
        best_b_map[instance_type][server_capacity] = b_devs_mean
        elite_configs = elite_configs_map[instance_type][server_capacity]
        elite_index = elite_configs.index
        config_ids = [a_devs_mean.index[0], b_devs_mean.index[0]]
        config_ids.extend(elite_index.astype(str).tolist())
        if instance_type not in best_config_ids_map:
            best_config_ids_map[instance_type] = {}
        best_config_ids_map[instance_type][server_capacity] = config_ids
        display(f"Instance: {instance_type}, Server Capacity: {server_capacity}")
        display(configs.loc[config_ids[:2]])

In [ ]:
# As best A and B configs don't have experiments run on all instances, we have run the best A and B configs on all instances separately
# The results are in `alt_exp_results` directory
for instance_type, server_capacities in full_experiments_map.items():
    for server_capacity, full_exps in server_capacities.items():
        # find the file that matches the pattern {instance_type}-{server_capacity}-A-*.csv and read it to a DataFrame
        pattern_a = f"alt_exp_results/{instance_type}-{server_capacity}-A-*.csv"
        pattern_b = f"alt_exp_results/{instance_type}-{server_capacity}-B-*.csv"
        matched_files_a = glob.glob(pattern_a)
        matched_files_b = glob.glob(pattern_b)
        if not matched_files_a:
            display(f"No files found for pattern {pattern_a}")
            continue
        if not matched_files_b:
            display(f"No files found for pattern {pattern_b}")
            continue
        alt_results_a = pd.read_csv(matched_files_a[0], index_col="instance")

        def index_sort_fn(x):
            # Sort numeric indices first, then non-numeric
            return (not x.isdigit(), int(x[:-1]) if x.endswith("t") else int(x))

        index_sorted_a = sorted(alt_results_a.index, key=index_sort_fn)
        alt_results_a = alt_results_a.reindex(index_sorted_a)
        alt_results_a.index = alt_results_a.index.astype(str)
        alt_results_b = pd.read_csv(matched_files_b[0], index_col="instance")
        index_sorted_b = sorted(alt_results_b.index, key=index_sort_fn)
        alt_results_b = alt_results_b.reindex(index_sorted_b)
        alt_results_b.index = alt_results_b.index.astype(str)
        best_configs = best_config_ids_map[instance_type][server_capacity]
        best_config_a = best_configs[0]
        best_config_b = best_configs[1]
        testing_exps = testing_exps_map[instance_type][server_capacity]
        testing_exps.index = testing_exps.index.astype(str)
        training_exps = training_exps_map[instance_type][server_capacity]
        training_exps.index = training_exps.index.astype(str)
        full_exps.index = full_exps.index.astype(str)
        full_exps[best_config_a] = alt_results_a["cost"]
        training_exps[best_config_a] = alt_results_a["cost"].iloc[:30]
        testing_exps[best_config_a] = alt_results_a["cost"].iloc[30:]
        full_exps[best_config_b] = alt_results_b["cost"]
        training_exps[best_config_b] = alt_results_b["cost"].iloc[:30]
        testing_exps[best_config_b] = alt_results_b["cost"].iloc[30:]
        devs = full_exps.copy()
        calc_dev(full_exps, devs)
        deviation_map[instance_type][server_capacity] = devs
        configs = configs_map[instance_type][server_capacity]
        algo_ids = {
            algo: configs[configs["algorithm"] == algo].index.astype(str).tolist()
            for algo in configs["algorithm"].unique()
        }
        cleanup_a_errors(full_exps, devs)
        # Extract A and B experiment and deviation data
        a_devs = devs[algo_ids["A"]]
        b_devs = devs[algo_ids["B"]]
        a_devs_mean = a_devs.mean(axis=0).sort_values()
        b_devs_mean = b_devs.mean(axis=0).sort_values()
        best_a_map[instance_type][server_capacity] = a_devs_mean
        best_b_map[instance_type][server_capacity] = b_devs_mean

In [ ]:
def config_summary_str(config_row, subindex=None):
    alg = config_row["algorithm"]
    if subindex is not None:
        result = f"{alg}<sub>{subindex}</sub>("
    else:
        result = f"{alg}("
    for col in config_row.index:
        if col == "algorithm":
            continue
        value = config_row[col]
        if pd.isna(value):
            continue
        if isinstance(value, (float, np.floating)):
            value = int(value)
        key = col
        if "MR" in col:
            key = "MR"
        result += f"{key}={value}, "
    return result[:-2] + ")"


summaries = {}
for instance_type, server_capacities in full_experiments_map.items():

    for server_capacity, full_exps in server_capacities.items():
        a_devs_mean = best_a_map[instance_type][server_capacity]
        b_devs_mean = best_b_map[instance_type][server_capacity]
        elite_configs = elite_configs_map[instance_type][server_capacity]
        elite_index = elite_configs.index
        devs = deviation_map[instance_type][server_capacity]
        config_ids = best_config_ids_map[instance_type][server_capacity]
        if server_capacity == "50":
            config_ids = config_ids[1:3]
        else:
            config_ids = config_ids[:3]
        # mean deviation
        # For most operations, A is skipped for server_capacity 50 as there isn't enough data to get accurate results, as algorithm A fails on most cases of server capacity 50
        if server_capacity == "50":
            mean_devs = [b_devs_mean.iloc[0]]
        else:
            mean_devs = [a_devs_mean.iloc[0], b_devs_mean.iloc[0]]
        mean_devs.append(devs[elite_index[0]].mean())
        mean_devs = np.array(mean_devs)

        # number of instances that config is run on
        # n_instances = [full_exps[idx].count() for idx in config_ids]

        # number of instances where deviation is 0 for an idx
        n_best = [0] * len(config_ids)
        best_devs = devs[config_ids]
        for row_idx, row in best_devs.iterrows():
            # get the column with the minimum value
            col_name = row.idxmin()
            # get the position of col_name in config_ids (a python list)
            idx = config_ids.index(col_name)
            n_best[idx] += 1

        # mean rank
        mean_rank = [0] * len(config_ids)
        best_exps = full_exps[config_ids]
        for row_idx, row in best_exps.iterrows():
            ranks = row.rank(method="min")
            for i, idx in enumerate(config_ids):
                rank = ranks[idx]
                if pd.notna(rank):
                    mean_rank[i] += rank
        mean_rank = np.array(mean_rank) / len(best_exps)

        configs = configs_map[instance_type][server_capacity]
        if server_capacity == "50":
            summary_config = [
                config_summary_str(configs.loc[config_ids[0]]),
                config_summary_str(configs.loc[elite_index[0]], subindex=1),
            ]
        else:
            summary_config = [
                config_summary_str(configs.loc[config_ids[0]]),
                config_summary_str(configs.loc[config_ids[1]]),
                config_summary_str(configs.loc[elite_index[0]], subindex=1),
            ]
        if instance_type not in summaries:
            summaries[instance_type] = {}
        summaries[instance_type][server_capacity] = {
            "Configuration": summary_config,
            "Config ID": config_ids,
            # "# Instances run on": n_instances,
            "Mean RPD": mean_devs,
            "# Best": n_best,
            "Mean Rank": mean_rank,
        }

display("Summary:")
summary = pd.DataFrame(
    columns=[
        "Instance type",
        "Server capacity",
        "Config ID",
        "Configuration",
        # "# Instances run on",
        "Mean RPD",
        "# Best",
        "Mean Rank",
    ]
)
for instance_type, server_capacities in summaries.items():
    for server_capacity, summary_data in server_capacities.items():
        n = len(summary_data["Configuration"])
        df = pd.DataFrame(
            {
                "Instance type": [instance_type] * n,
                "Server capacity": [server_capacity] * n,
                **summary_data,
            }
        )
        summary = pd.concat([summary, df], ignore_index=True)
display(summary.style.hide(axis="index"))

In [ ]:
summaries_elites = {}
for instance_type, server_capacities in full_experiments_map.items():

    for server_capacity, full_exps in server_capacities.items():
        a_devs_mean = best_a_map[instance_type][server_capacity]
        b_devs_mean = best_b_map[instance_type][server_capacity]
        elite_configs = elite_configs_map[instance_type][server_capacity]
        elite_index = elite_configs.index
        devs = deviation_map[instance_type][server_capacity]
        config_ids = best_config_ids_map[instance_type][server_capacity][2:]
        # mean deviation
        # For most operations, A is skipped for server_capacity 50 as there isn't enough data to get accurate results, as algorithm A fails on most cases of server capacity 50
        mean_devs = []
        for idx in elite_index:
            mean_devs.append(devs[idx].mean())
        mean_devs = np.array(mean_devs)

        # number of instances that config is run on
        # n_instances = [full_exps[idx].count() for idx in config_ids]

        # number of instances where deviation is 0 for an idx
        n_best = [0] * len(config_ids)
        best_devs = devs[config_ids]
        for row_idx, row in best_devs.iterrows():
            # get the column with the minimum value
            col_name = row.idxmin()
            # get the position of col_name in config_ids (a python list)
            idx = config_ids.index(col_name)
            n_best[idx] += 1

        # mean rank
        mean_rank = [0] * len(config_ids)
        best_exps = full_exps[config_ids]
        for row_idx, row in best_exps.iterrows():
            ranks = row.rank(method="min")
            for i, idx in enumerate(config_ids):
                rank = ranks[idx]
                if pd.notna(rank):
                    mean_rank[i] += rank
        mean_rank = np.array(mean_rank) / len(best_exps)

        configs = configs_map[instance_type][server_capacity]
        summary_config = [
            config_summary_str(configs.loc[idx], subindex=i + 1)
            for i, idx in enumerate(config_ids)
        ]
        if instance_type not in summaries_elites:
            summaries_elites[instance_type] = {}
        summaries_elites[instance_type][server_capacity] = {
            "Configuration": summary_config,
            "Config ID": config_ids,
            # "# Instances run on": n_instances,
            "Mean RPD": mean_devs,
            "# Best": n_best,
            "Mean Rank": mean_rank,
        }

display("Summary:")
summary = pd.DataFrame(
    columns=[
        "Instance type",
        "Server capacity",
        "Config ID",
        "Configuration",
        # "# Instances run on",
        "Mean RPD",
        "# Best",
        "Mean Rank",
    ]
)
for instance_type, server_capacities in summaries_elites.items():
    for server_capacity, summary_data in server_capacities.items():
        n = len(summary_data["Configuration"])
        df = pd.DataFrame(
            {
                "Instance type": [instance_type] * n,
                "Server capacity": [server_capacity] * n,
                **summary_data,
            }
        )
        summary = pd.concat([summary, df], ignore_index=True)
display(summary.style.hide(axis="index"))

In [ ]:
# The first elite config is the best one considered by irace, so we run each best elite config on all combinations of instance types and server capacities
# (running them only on test instances)
best_elite_runs = pd.DataFrame(
    columns=[
        "Instance type",
        "Server capacity",
        "Was best elite",
        "Instance",
        "Configuration",
        "Cost",
        # "Time"
    ]
)

best_elites = {
    "C1(MR=2, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=LL, NVSP=HL, PSP=RR)": [
        ("small", "50"),
        ("medium", "50"),
    ],
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=RND, NVSP=LL, PSP=RR)": (
        "small",
        "150",
    ),
    "C1(MR=2, RT=SES, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=LL, PSP=LL)": (
        "small",
        "650",
    ),
    "C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=IT, OVSP=LL, NVSP=HL, PSP=LL)": (
        "small",
        "1000",
    ),
    "C1(MR=3, RT=SER, EVST=TTB, EVSP=LL, OVST=BTT, OVSP=HL, NVSP=LL, PSP=RR)": (
        "medium",
        "150",
    ),
    "C1(MR=6, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=HL, PSP=RND)": (
        "medium",
        "650",
    ),
    "C1(MR=9, RT=SER, EVST=TTB, EVSP=LL, OVST=TTB, OVSP=LL, NVSP=LL, PSP=LL)": (
        "medium",
        "1000",
    ),
}


def parse_log(file, instance_type):
    with open(file, "r") as f:
        df_row = {
            "Instance type": instance_type,
            "Server capacity": None,
            "Was best elite": None,
            "Configuration": None,
            "Instance": None,
            "Cost": None,
        }
        for line in f:
            if "Output for" in line:
                # Find substring CAPACITY= and read the number after it, preceeding the ,
                server_capacity = line.split("CAPACITY=")[1].split(",")[0]
                instance = line.split(",")[1].split("=")[1]
                real_instance = str(int(instance) - 29) + "t"
                config = line.split("args=")[1].strip()
                best_config = best_elites.get(config, None)
                if isinstance(best_config, list):
                    for best in best_config:
                        df_row["Was best elite"] = "F"
                        if best[0] == instance_type and best[1] == server_capacity:
                            df_row["Was best elite"] = "T"
                            break
                else:
                    if (
                        best_config[0] == instance_type
                        and best_config[1] == server_capacity
                    ):
                        df_row["Was best elite"] = "T"
                    else:
                        df_row["Was best elite"] = "F"
                df_row["Server capacity"] = int(server_capacity)
                df_row["Instance"] = real_instance
                df_row["Configuration"] = config
            else:
                cost = float(line.split(" ")[0])
                df_row["Cost"] = cost
                best_elite_runs.loc[len(best_elite_runs)] = df_row


parse_log("test_elite_configs/small.log", "small")
parse_log("test_elite_configs/medium.log", "medium")
display(
    best_elite_runs.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)

In [ ]:
# For each combination of instance type, server capacity and instance in best_elite_runs, calculate the RPD of all configurations
instance_types = best_elite_runs["Instance type"].unique()
server_capacities = best_elite_runs["Server capacity"].unique()
instances = best_elite_runs["Instance"].unique()
for instance_type in instance_types:
    for server_capacity in server_capacities:
        for instance in instances:
            best_elites = best_elite_runs[
                (best_elite_runs["Instance type"] == instance_type)
                & (best_elite_runs["Server capacity"] == server_capacity)
                & (best_elite_runs["Instance"] == instance)
            ]
            ranks = best_elites["Cost"].rank(method="min")
            best_elite_runs.loc[
                (best_elite_runs["Instance type"] == instance_type)
                & (best_elite_runs["Server capacity"] == server_capacity)
                & (best_elite_runs["Instance"] == instance),
                "Rank",
            ] = ranks
            best_elite_run = best_elites.sort_values(by="Cost").iloc[0]
            best_elite_config = best_elite_run["Configuration"]
            best_elite_cost = best_elite_run["Cost"]
            for idx_row, row in best_elites.iterrows():
                config = row["Configuration"]
                cost = row["Cost"]
                rpd = 100 * abs((cost - best_elite_cost) / best_elite_cost)
                best_elite_runs.loc[
                    (best_elite_runs["Instance type"] == instance_type)
                    & (best_elite_runs["Server capacity"] == server_capacity)
                    & (best_elite_runs["Instance"] == instance)
                    & (best_elite_runs["Configuration"] == config)
                    & (best_elite_runs["Cost"] == cost),
                    "RPD",
                ] = rpd.round(3)
                best_elite_runs.loc[
                    (best_elite_runs["Instance type"] == instance_type)
                    & (best_elite_runs["Server capacity"] == server_capacity)
                    & (best_elite_runs["Instance"] == instance)
                    & (best_elite_runs["Configuration"] == config)
                    & (best_elite_runs["Cost"] == cost),
                    "Is best",
                ] = (
                    cost == best_elite_cost
                )

display(
    best_elite_runs.sort_values(
        by=["Instance type", "Server capacity", "Instance"],
        ascending=[False, True, True],
    )
)

In [ ]:
def group_alt_results(group_array, sort_by=None, ascending=None):
    avgs = best_elite_runs.groupby(group_array)["RPD"].mean().round(3).reset_index()
    avgs = avgs.rename(columns={"RPD": "Mean RPD"})

    max_rpd = best_elite_runs.groupby(group_array)["RPD"].max().round(3).reset_index()
    avgs = avgs.merge(
        max_rpd,
        on=group_array,
    )
    avgs = avgs.rename(columns={"RPD": "Max RPD"})

    q1 = (
        best_elite_runs.groupby(group_array)["RPD"]
        .quantile(0.25)
        .round(3)
        .reset_index()
    )
    avgs = avgs.merge(q1, on=group_array)
    avgs = avgs.rename(columns={"RPD": "Q1 RPD"})
    median = best_elite_runs.groupby(group_array)["RPD"].median().round(3).reset_index()
    avgs = avgs.merge(
        median,
        on=group_array,
    )
    avgs = avgs.rename(columns={"RPD": "Median RPD"})
    q3 = (
        best_elite_runs.groupby(group_array)["RPD"]
        .quantile(0.75)
        .round(3)
        .reset_index()
    )
    avgs = avgs.merge(q3, on=group_array)
    avgs = avgs.rename(columns={"RPD": "Q3 RPD"})

    best_counts = (
        best_elite_runs.groupby(group_array)["Is best"]
        .apply(lambda x: x.eq(True).sum())
        .reset_index(name="# Best")
    )
    avgs = avgs.merge(
        best_counts,
        on=group_array,
    )
    avgs["Mean Rank"] = (
        best_elite_runs.groupby(group_array)["Rank"]
        .mean()
        .round(3)
        .reset_index(drop=True)
    )

    return avgs.sort_values(by=sort_by, ascending=ascending)


# convert to tex table
def to_latex_table(df, filename):
    with open(filename, "w") as f:
        f.write(
            df.to_latex(
                index=False,
                float_format="%.3f",
                column_format="l" + "c" * (len(df.columns) - 1),
                escape=False,
            )
        )

avgs = group_alt_results(
    ["Instance type", "Server capacity", "Was best elite", "Configuration"],
    sort_by=["Instance type", "Server capacity", "Configuration"],
    ascending=[False, True, True],
)
display(avgs)
to_latex_table(avgs, filename="alt_results.tex")

In [ ]:
avgs_server = group_alt_results(
    ["Server capacity", "Configuration"],
    sort_by=["Server capacity", "Configuration"],
    ascending=[True, True],
)

display(avgs_server)
to_latex_table(avgs_server, filename="alt_results_server.tex")

In [ ]:
avgs_instance = group_alt_results(
    ["Instance type", "Configuration"],
    sort_by=["Instance type", "Configuration"],
    ascending=[False, True],
)

display(avgs_instance)
to_latex_table(avgs_instance, filename="alt_results_instance.tex")

In [ ]:
avgs_all = group_alt_results(
    ["Configuration"],
    sort_by=["Configuration"],
    ascending=[True],
)
display(avgs_all)
to_latex_table(avgs_all, filename="alt_results_all.tex")

In [ ]:
def run_stat_tests(
    group_label_1,
    group_label_2,
    friedmann_tests,
    wilcoxon_tests,
    full_exps,
    best_config_ids,
):
    common_indices = full_exps.index
    for config_id in best_config_ids:
        common_indices = common_indices.intersection(
            full_exps[config_id].dropna().index
        )
    if common_indices.empty:
        return
    friedmann_samples = [
        full_exps[config_id].loc[common_indices] for config_id in best_config_ids
    ]

    stat, p_value = friedmanchisquare(*friedmann_samples)

    friedmann_tests.loc[len(friedmann_tests)] = {
        "Instance Type": group_label_1,
        "Server Capacity": group_label_2,
        "statistic": stat,
        "p-value": p_value,
    }

    labels = {config_id: "" for config_id in best_config_ids}
    labels[best_config_ids[0]] = "Best A"
    labels[best_config_ids[1]] = "Best B"
    for i, config_id in enumerate(best_config_ids[2:]):
        labels[config_id] = f"Elite config {i+1}"

    for config1, config2 in combinations(best_config_ids, 2):

        common_indices = (
            full_exps[config1]
            .dropna()
            .index.intersection(full_exps[config2].dropna().index)
        )
        config1_sample = full_exps[config1].loc[common_indices]
        config2_sample = full_exps[config2].loc[common_indices]

        diff = config1_sample - config2_sample


        # Does the medians of the differences differ?


        # If the p value is less than 0.05, then config2 is significantly better than config1
        if not (diff == 0).all():

            stat, p_value = wilcoxon(diff.round(3), alternative="greater")

            wilcoxon_tests.loc[len(wilcoxon_tests)] = {
                "Instance Type": group_label_1,
                "Server Capacity": group_label_2,
                "Config 1": labels[config1],
                "Config 2": labels[config2],
                "W": stat,
                "p-value": p_value,
            }



friedmann_tests = pd.DataFrame(
    columns=["Instance Type", "Server Capacity", "statistic", "p-value"]
)


wilcoxon_tests = pd.DataFrame(
    columns=[
        "Instance Type",
        "Server Capacity",
        "Config 1",
        "Config 2",
        "W",
        "p-value",
    ]
)



for instance_type, server_capacities in full_experiments_map.items():

    for server_capacity, full_exps in server_capacities.items():

        run_stat_tests(
            instance_type,
            server_capacity,
            friedmann_tests,
            wilcoxon_tests,
            full_exps,
            best_config_ids_map[instance_type][server_capacity],
        )



display("Friedmann Tests Results")


display(friedmann_tests.style.hide(axis="index"))


friedmann_tests.to_csv("friedmann_tests_results.csv", index=False)


display("Wilcoxon Tests Results")


display(wilcoxon_tests.style.hide(axis="index"))


wilcoxon_tests.to_csv("wilcoxon_tests_results.csv", index=False)

In [ ]:
merged_map = {}


def calc_deviations(instance_type, server_capacity, merged):
    deviation_df = merged.copy()
    deviation_df["min_objective_per_instance"] = deviation_df.groupby("instance")[
        "objective"
    ].transform("min")
    # Use percentage deviation
    deviation_df["objective"]
    deviation_df["min_objective_per_instance"]
    deviation_df["RPD"] = 100 * abs(
        (deviation_df["objective"] - deviation_df["min_objective_per_instance"])
        / deviation_df["min_objective_per_instance"]
    )

    mean_deviation_per_instance = (
        deviation_df.groupby("instance")["RPD"].mean().sort_index()
    )

    elite_configs = elite_configs_map[instance_type][server_capacity]
    elite_merged = merged[merged["config_id"].isin(elite_configs.index.astype(str))]
    elite_deviation_df = elite_merged.copy()
    elite_deviation_df["min_objective_per_instance"] = deviation_df.groupby("instance")[
        "objective"
    ].transform("min")
    elite_deviation_df["RPD"] = 100 * abs(
        (
            elite_deviation_df["objective"]
            - elite_deviation_df["min_objective_per_instance"]
        )
        / elite_deviation_df["min_objective_per_instance"]
    )
    return mean_deviation_per_instance, elite_deviation_df


def summarize_deviations(mean_deviation_all, elite_deviation_all):
    # Calculate summary statistics for mean deviation per instance (all configs)
    summary_df = pd.DataFrame(
        {
            "Avg Mean Deviation (%) (All)": [mean_deviation_all.mean()],
        }
    )
    summary_df.index = ["All Configs"]

    elite_config_ids = elite_deviation_all["config_id"].unique()
    # Calculate summary statistics for each elite config individually
    elite_config_stats = []
    for i in range(len(elite_config_ids)):
        config_id = elite_config_ids[i]
        df_all = elite_deviation_all[elite_deviation_all["config_id"] == config_id]
        all_per_instance = df_all.groupby("instance")["RPD"].mean().sort_index()

        elite_config_stats.append(
            {
                "config_id": "Elite Config " + str(i + 1),
                "Avg Mean Deviation (%) (All)": all_per_instance.mean(),
            }
        )
    if elite_config_stats:
        elite_config_stats_df = pd.DataFrame(elite_config_stats).set_index("config_id")
        elite_config_stats_df.index.name = "Elite Config ID"
    # Round and display all together
    display(
        f"Mean Deviation (%) Summary Statistics for instance type {instance_type} and server capacity {server_capacity}:"
    )
    display(pd.concat([summary_df, elite_config_stats_df]).round(3))


for instance_type, server_capacities in full_experiments_map.items():
    for server_capacity, full_exps in server_capacities.items():
        configs = configs_map[instance_type][server_capacity]
        # only keep the columns that are in configs
        c_configs = configs[configs["algorithm"] == "C"].dropna(axis=1, how="all")
        c_configs = c_configs.drop(columns=["algorithm"]).rename(
            columns={"MR [C]": "MR"}
        )
        e = full_exps[c_configs.index]

        # Prepare data for ANOVA: melt experiments to long format and merge with parameter values
        exp_long = e.reset_index().melt(
            id_vars="index", var_name="config_id", value_name="objective"
        )
        exp_long = exp_long.rename(columns={"index": "instance"})
        exp_long["config_id"] = exp_long["config_id"].astype(str)
        c_configs_ = c_configs.copy()
        c_configs_["config_id"] = c_configs_.index.astype(str)
        merged = exp_long.merge(c_configs_, on="config_id")
        merged["MR"] = merged["MR"].astype(int)

        mean_deviation_per_instance, elite_deviation_df = calc_deviations(
            instance_type, server_capacity, merged
        )

        summarize_deviations(
            mean_deviation_per_instance,
            elite_deviation_df,
        )

        if instance_type not in merged_map:
            merged_map[instance_type] = {}
        merged_map[instance_type][server_capacity] = merged

In [ ]:
for instance_type, server_capacities in training_exps_map.items():
    for server_capacity, full_exps in server_capacities.items():
        merged = merged_map[instance_type][server_capacity]
        df_clean = merged.dropna(subset=["objective"]).copy()

        # Get elite config ids for this instance/capacity
        elite_configs = elite_configs_map[instance_type][server_capacity]
        elite_configs = elite_configs.rename(columns={"MR [C]": "MR"})
        elite_ids = set(elite_configs.index.astype(str))

        # Compute deviation from the best for each instance
        df_clean.loc[:, "min_objective_per_instance"] = df_clean.groupby("instance")[
            "objective"
        ].transform("min")
        df_clean.loc[:, "Mean RPD"] = 100 * abs(
            (df_clean["objective"] - df_clean["min_objective_per_instance"])
            / df_clean["min_objective_per_instance"]
        )

        # Compute and display mean deviation from best for each parameter in the data
        param_cols = df_clean.columns.difference(
            [
                "instance",
                "config_id",
                "objective",
                "min_objective_per_instance",
                "Mean RPD",
            ]
        )
        for param in param_cols:
            mean_deviation_param = (
                df_clean.groupby(param)["Mean RPD"].mean().sort_values()
            )

            # For each parameter value, find elite config ids that chose it
            elite_param_map = {}
            for value in mean_deviation_param.index:
                elite_ids_with_value = (
                    elite_configs[elite_configs[param] == value]
                    .index.astype(str)
                    .tolist()
                )
                elite_param_map[value] = (
                    elite_ids_with_value if elite_ids_with_value else None
                )

            # Create a DataFrame for display
            result_df = mean_deviation_param.round(3).to_frame("Mean RPD")
            result_df["In Elite"] = result_df.index.map(lambda v: elite_param_map[v])

            display(
                f"Mean RPD for {instance_type}, Server Capacity {server_capacity} by {param}:"
            )
            display(result_df)

In [ ]:
# Scatterplot: MR vs objective, colored by RT
for instance_type, server_capacities in training_exps_map.items():
    for server_capacity, full_exps in server_capacities.items():
        merged = merged_map[instance_type][server_capacity]
        for col in merged.columns:
            if col not in ["instance", "objective", "config_id"]:
                fig, axs = plt.subplots(figsize=(24, 30), ncols=3, nrows=10)
                unique_instances = merged["instance"].unique()
                axs_flat = axs.flatten()
                fig.suptitle(f"Scatterplot of {col} vs Objective", y=0.98)
                for i, instance in enumerate(unique_instances):
                    ax = axs_flat[i]
                    sns.scatterplot(
                        data=merged[merged["instance"] == instance],
                        x=col,
                        y="objective",
                        ax=ax,
                    )
                    ax.set_title(
                        f"Instance {instance} ({instance_type}, Server Capacity {server_capacity})"
                    )
                    ax.set_xlabel(col)
                    ax.set_ylabel("Objective")
                # Hide unused axes
                for j in range(len(unique_instances), len(axs_flat)):
                    axs_flat[j].set_visible(False)
                plt.tight_layout(rect=[0, 0, 1, 0.96])
                plt.subplots_adjust(top=0.93)
        break

In [ ]:
# testing_exp is a DataFrame where each column represents a parameter configuration, each row an instance.
# The cell contains the value of the objective function for that configuration ran on that instance.
def calculate_statistics(testing_exp):
    # Calculate mean, std, and 95% CI for each configuration (column), taking into account all instances (rows)
    means = testing_exp.mean(axis=0)
    stds = testing_exp.std(axis=0, ddof=1)
    n = testing_exp.shape[0]
    ci95 = 1.96 * stds / (n**0.5)

    stats = pd.DataFrame({"Mean": means, "Std Dev": stds, "95% CI": ci95})

    # Sort by mean (assuming lower is better; change to ascending=False if higher is better)
    stats = stats.sort_values("Mean")
    return stats


for instance_type, server_capacities in testing_exps_map.items():
    for server_capacity, test_exp in server_capacities.items():
        stats = calculate_statistics(test_exp)
        display(f"Instance Type: {instance_type}, Server Capacity: {server_capacity}")
        display(stats)

In [ ]:
# calculate chosen parameter frequency
def calculate_chosen_parameter_frequency(configs):
    frequency = {}
    for col in configs.columns:
        if col == "algorithm" or col in alg_c_parameters:
            if col not in frequency:
                frequency[col] = {}
            for value in configs[col]:
                if value is None or (isinstance(value, float) and pd.isna(value)):
                    continue
                if isinstance(value, float) and value.is_integer():
                    value = int(value)
                if value not in frequency[col]:
                    frequency[col][value] = 0
                frequency[col][value] += 1
    return frequency


# for every instance type and server capacity, calculate the chosen parameter frequency
chosen_parameter_frequency_map = {}
for instance_type, server_capacities in configs_map.items():
    for server_capacity, config in server_capacities.items():
        if instance_type not in chosen_parameter_frequency_map:
            chosen_parameter_frequency_map[instance_type] = {}
        frequency = calculate_chosen_parameter_frequency(config)
        chosen_parameter_frequency_map[instance_type][server_capacity] = frequency
# for every instance type and server capacity, display the chosen parameter frequency
# Use subplots for every instance type and server capacity
for instance_type, server_capacities in chosen_parameter_frequency_map.items():
    for server_capacity, frequency in server_capacities.items():
        all_params = ["algorithm"] + sorted(
            [p for p in frequency.keys() if p != "algorithm"]
        )
        n_params = len(all_params)
        ncols = 3
        nrows = (n_params - 1 + ncols) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows))
        fig.suptitle(
            f"Parameter Frequency for {instance_type} - Server capacity {server_capacity}\n(Only parameters used by C algorithm are shown)",
        )
        axes = axes.flatten()
        for idx, param in enumerate(all_params):
            param_values = frequency[param]
            if param_values:
                freq_series = pd.Series(param_values)
                # Check if all keys are ints (or can be safely cast to int)
                try:
                    keys_as_int = [int(k) for k in freq_series.index]
                    is_all_int = all(
                        isinstance(k, int) or (isinstance(k, float) and k.is_integer())
                        for k in freq_series.index
                    )
                except Exception:
                    is_all_int = False
                    freq_series = freq_series.sort_index()
                if is_all_int and len(freq_series) > 1:
                    # Expand the series for histogram plotting
                    expanded = []
                    for k, v in freq_series.items():
                        expanded.extend([int(k)] * v)
                    axes[idx].hist(expanded, bins=20, edgecolor="black")
                    axes[idx].set_title(f"{param} histogram")
                    axes[idx].set_ylabel("Frequency")
                else:
                    sns.barplot(
                        x=freq_series.index.astype(str),
                        y=freq_series.values,
                        ax=axes[idx],
                        edgecolor="black",
                    )
                    axes[idx].set_title(f"{param} frequency")
                    axes[idx].set_xlabel("Parameter Value")
                    axes[idx].set_ylabel("Frequency")
            else:
                axes[idx].set_visible(False)
        for idx in range(len(all_params), len(axes)):
            axes[idx].set_visible(False)
        plt.tight_layout(rect=[0, 0, 1, 0.96])

In [ ]:
# heatmap of server capacity and max reservation in elite configs
heatmap_data = []
for instance_type, server_capacities in elite_configs_map.items():
    for server_capacity, elite_config in server_capacities.items():
        if "MR [C]" in elite_config.columns:
            heatmap_data.append(
                {
                    "Instance Type": instance_type,
                    "Server Capacity": server_capacity,
                    "Max Reservation": elite_config[elite_config["RT"] == "SER"][
                        "MR [C]"
                    ].mean(),
                }
            )
        if "MR" in elite_config.columns:
            heatmap_data.append(
                {
                    "Instance Type": instance_type,
                    "Server Capacity": server_capacity,
                    "Max Reservation": elite_config[elite_config["RT"] == "SER"][
                        "MR"
                    ].mean(),
                }
            )

heatmap_df = pd.DataFrame(heatmap_data)
heatmap_pivot = heatmap_df.pivot(
    index="Instance Type", columns="Server Capacity", values="Max Reservation"
)

# Sort columns numerically for correct order in heatmap
sorted_columns = sorted(heatmap_pivot.columns, key=lambda x: int(x))
heatmap_pivot = heatmap_pivot[sorted_columns]

plt.figure(figsize=(8, 4))
sns.heatmap(heatmap_pivot, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title(
    "Heatmap of Average Max Reservation by Instance Type and Server Capacity\non Elite Configurations (Only Reservation Per Server)"
)
plt.ylabel("Instance Type")
plt.xlabel("Server Capacity")